# Pneumonia Detection from Chest X-Rays

### Overview

Pneumonia is often diagnosed from chest X-ray images. The goal of this project is a model that classifies a chest X-ray as **pneumonia** or **normal**, using deep learning and transfer learning.

Because medical image datasets are small, a network trained from scratch tends to overfit. The strongest approach is transfer learning: reusing a model already trained on millions of general images and adapting it to X-rays. Accuracy alone is not a sufficient metric; in medicine a missed case (low recall) is the costly error, so recall is the priority.

### Steps taken

1. Download and load the images from folders.
2. Split into training, validation, and test sets.
3. Preprocess and augment the images.
4. Visualize sample X-rays.
5. Train a simple CNN as a baseline.
6. Train a stronger model using transfer learning.
7. Apply dropout and early stopping to reduce overfitting.
8. Plot training and validation curves.
9. Tune key hyperparameters.
10. Evaluate with recall, ROC-AUC, and a confusion matrix.
11. Explain predictions with Grad-CAM.

---

## Where the data comes from

The dataset is **Chest X-Ray Images (Pneumonia)**, from a peer-reviewed study (Kermany et al., *Cell*, 2018) using real pediatric chest X-rays. It was obtained from **Kaggle**.

The notebook connects to Kaggle once using a small key file (`kaggle.json`) and downloads the dataset automatically.

**How to get the key file (one time):**
1. Go to https://www.kaggle.com/settings
2. Scroll down to the **API** section.
3. Click **Create New API Token**.
4. The `kaggle.json` file downloads automatically to the **Downloads** folder.

**Also enable the GPU** in Colab for faster training: Runtime > Change runtime type > GPU.

Run the cell below: when the **Choose Files** button appears, select `kaggle.json` from the Downloads folder.

In [ ]:
# Downloads the dataset from Kaggle. Click 'Choose Files' and select kaggle.json.
from google.colab import files
files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q -o chest-xray-pneumonia.zip
print('The chest_xray folder is ready.')

## Step 0 — Libraries

TensorFlow and the supporting libraries are imported below. A fixed random seed makes the run reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Early stopping is used by every training step, so it is defined once here.
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## Step 1 — Load the images

The images are stored in folders (`chest_xray/train`, `chest_xray/val`, `chest_xray/test`), each containing `NORMAL` and `PNEUMONIA` subfolders. Keras can build datasets directly from this structure. Images are resized to 224x224 (the size the pretrained model expects) and grouped into batches.

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32
BASE = 'chest_xray'

train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{BASE}/train', image_size=IMG_SIZE, batch_size=BATCH, label_mode='binary', seed=SEED)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{BASE}/val', image_size=IMG_SIZE, batch_size=BATCH, label_mode='binary', seed=SEED)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f'{BASE}/test', image_size=IMG_SIZE, batch_size=BATCH, label_mode='binary', seed=SEED)

class_names = train_ds.class_names
print('Classes:', class_names)

The built-in validation folder is very small. A portion of the training data is set aside for a more reliable validation signal. Prefetching is enabled so the GPU is not kept waiting for data.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## Step 2 — Visualize sample X-rays

Viewing a few images confirms the data loaded correctly and shows what the two classes look like.

In [ ]:
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(6):
        plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        label = class_names[int(labels[i][0])]
        plt.title(label)
        plt.axis('off')
plt.tight_layout()
plt.savefig('sample_images.png', bbox_inches='tight', dpi=150)
plt.show()

## Step 3 — Preprocessing and augmentation

Two layers are defined. Rescaling converts pixel values to the 0-1 range. Augmentation randomly flips and rotates training images, which helps the model generalise and reduces overfitting on a small dataset.

In [ ]:
data_augmentation = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

## Step 4 — Baseline model (a small CNN)

A simple convolutional network trained from scratch sets the baseline score that the transfer-learning model must beat.

In [ ]:
baseline = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(16, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.3),                      # dropout reduces overfitting
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid'),    # one output: probability of pneumonia
])
baseline.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_base = baseline.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stop])

## Step 5 — Transfer learning (the stronger model)

**Why transfer learning, and why freeze then fine-tune?**

The choice of strategy depends on two things: the size of the dataset and how similar the task is to the pretrained task.

- The pneumonia dataset is **small** (about 5,800 images), so training a deep network from scratch would overfit. Reusing a network pretrained on millions of images (ImageNet) gives a large head start.
- Chest X-rays are **visually quite different** from ImageNet's everyday photos. The lower layers of a pretrained network capture general features (edges, textures) that still transfer well, while the higher layers are more task-specific.

Given a small dataset and a somewhat different task, the safe and effective strategy is:
1. **Freeze** all pretrained layers and train only a new classifier head. This reuses the general features and avoids overfitting on limited data.
2. Later, **fine-tune** only the top block with a very small learning rate (Step 7), to adapt the higher-level features to X-rays without erasing the useful general features in the lower layers.

The step below performs the first part: a **ResNet50** backbone pretrained on ImageNet, frozen, with a small new classifier head trained on top.

In [ ]:
base_model = ResNet50(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base_model.trainable = False   # freeze the pretrained layers

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.resnet50.preprocess_input(x)   # ResNet's own scaling
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = models.Model(inputs, outputs)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stop])

## Step 6 — Training and validation curves

Plotting loss and accuracy over the training epochs shows whether the model is learning well and whether it is overfitting (training score improving while validation score stalls).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='Training loss')
ax1.plot(history.history['val_loss'], label='Validation loss')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history.history['accuracy'], label='Training accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation accuracy')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.savefig('training_curves.png', bbox_inches='tight', dpi=150)
plt.show()

## Step 7 — Hyperparameter tuning (fine-tuning)

Fine-tuning unfreezes the top layers of the pretrained network and continues training with a **much smaller learning rate**. The learning rate is the key hyperparameter here: too high would destroy the pretrained knowledge, so a small value is used. This often gives a further improvement over the frozen model.

In [ ]:
base_model.trainable = True
# Keep the earliest layers frozen; fine-tune only the last block
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # small LR for fine-tuning
    loss='binary_crossentropy', metrics=['accuracy'])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[early_stop])

## Step 8 — Evaluation

The model is evaluated on the held-out test set. Recall (how many real pneumonia cases were caught), ROC-AUC, and the confusion matrix matter more than plain accuracy.

In [ ]:
# Collect true labels and predicted probabilities over the test set
y_true, y_prob = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_prob.extend(probs.ravel())
    y_true.extend(labels.numpy().ravel())
y_true = np.array(y_true)
y_pred = (np.array(y_prob) >= 0.5).astype(int)

print('Test ROC-AUC:', round(roc_auc_score(y_true, y_prob), 3))
print()
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion matrix (test set)')
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

## Step 9 — Explainability with Grad-CAM

Grad-CAM highlights the regions of an X-ray that most influenced the model's decision, shown as a heatmap over the image. This makes the model inspectable rather than a black box.

In [ ]:
# Grad-CAM needs the last conv layer of ResNet50, which is nested inside the model.
# A single grad model is built that goes: image -> conv feature maps -> final prediction,
# so the gradient can flow end to end in one forward pass.
resnet = model.get_layer('resnet50')
last_conv = resnet.get_layer('conv5_block3_out')

# Sub-model: ResNet input -> last conv feature maps
conv_submodel = tf.keras.models.Model(resnet.input, last_conv.output)

def grad_cam(img_array):
    img_array = tf.cast(img_array, tf.float32)
    x = tf.keras.applications.resnet50.preprocess_input(img_array)
    with tf.GradientTape() as tape:
        conv_out = conv_submodel(x)          # conv feature maps
        tape.watch(conv_out)
        # Continue manually through the model head on top of conv_out
        pooled_feat = tf.reduce_mean(conv_out, axis=(1, 2))      # global avg pool
        # Use the trained Dense head weights from the full model
        dense = model.get_layer('dense_3') if 'dense_3' in [l.name for l in model.layers] else model.layers[-1]
        preds = dense(pooled_feat)
        loss = preds[:, 0]
    grads = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Take one test image and overlay the heatmap
for images, labels in test_ds.take(1):
    img = images[0:1]
    heatmap = grad_cam(img)
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1); plt.imshow(images[0].numpy().astype('uint8')); plt.title('X-ray'); plt.axis('off')
    plt.subplot(1, 2, 2); plt.imshow(images[0].numpy().astype('uint8'))
    plt.imshow(tf.image.resize(heatmap[..., None], IMG_SIZE).numpy().squeeze(), cmap='jet', alpha=0.5)
    plt.title('Grad-CAM'); plt.axis('off')
    plt.savefig('grad_cam.png', bbox_inches='tight', dpi=150)
    plt.show()
    break